# Leo: Multi-Agent AI Tutor
**Assignment project — CrewAI + Gemini API**

This notebook implements the four required roles, sequential handoffs, student/topic memory, and the bonus re-teaching loop.

**Important:** Use a Gemini API key from Google AI Studio. Do not paste it into a code cell or commit it to GitHub.


## 1. Install packages
Run this cell first. When installation finishes, if Colab asks for a runtime restart, restart and then continue from cell 2.

In [ ]:
%pip -q install -U "crewai[google-genai]>=1.15.0" "google-genai>=1.0.0"


## 2. Imports and secure Gemini API key
The key is entered through a hidden prompt. It is not saved in the notebook.

In [ ]:
import os
import json
import re
from pathlib import Path
from getpass import getpass

from google import genai
from crewai import Agent, Crew, LLM, Process, Task

# Clear any previous OpenAI key from this runtime.
os.environ.pop("OPENAI_API_KEY", None)

GEMINI_API_KEY = getpass("Paste Gemini API key (hidden): ").strip()
if not GEMINI_API_KEY:
    raise ValueError("Gemini API key is empty.")

os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

MODEL_NAME = "gemini-2.5-flash-lite"
CREWAI_MODEL = f"gemini/{MODEL_NAME}"
MEMORY_FILE = Path("/content/leo_memory.json")

print("Gemini key loaded into this runtime. The key will not be displayed.")


## 3. Test Gemini API access before running agents
If this test fails, the issue is the key, API access, quota, or model availability—not CrewAI.

In [ ]:
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
test_response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents="Reply with exactly: Gemini API connection successful."
)
print(test_response.text)


## 4. Student memory and model configuration

In [ ]:
def load_memory():
    if MEMORY_FILE.exists():
        try:
            return json.loads(MEMORY_FILE.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            return {}
    return {}

def save_memory(student_name, topic, level, score, feedback):
    memory = load_memory()
    memory["student_name"] = student_name
    memory["last_topic"] = topic
    memory.setdefault("sessions", []).append({
        "topic": topic,
        "level": level,
        "score": score,
        "feedback_summary": feedback[:1200],
    })
    MEMORY_FILE.write_text(
        json.dumps(memory, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

llm = LLM(
    model=CREWAI_MODEL,
    api_key=GEMINI_API_KEY,
    temperature=0.3,
)

print("CrewAI Gemini LLM configured:", CREWAI_MODEL)
print("Remembered student:", load_memory().get("student_name", "None yet"))
print("Last topic:", load_memory().get("last_topic", "None yet"))


## 5. Create four specialised agents
Each agent has a separate role, goal, and backstory prompt.

In [ ]:
def build_agents():
    coordinator = Agent(
        role="Learning Coordinator",
        goal="Understand the student's request and create a focused learning plan.",
        backstory=(
            "You coordinate Leo's tutoring team. Identify the student's level, "
            "break the topic into clear objectives, and pass a useful plan to the teacher."
        ),
        llm=llm, verbose=True, allow_delegation=False, max_iter=3,
    )
    explainer = Agent(
        role="Concept Explainer",
        goal="Teach the requested topic accurately and clearly.",
        backstory=(
            "You are a patient teacher. Explain ideas step by step in student-friendly "
            "language, define key terms, use a practical example, and recap the lesson."
        ),
        llm=llm, verbose=True, allow_delegation=False, max_iter=3,
    )
    quiz_master = Agent(
        role="Quiz Master",
        goal="Create a structured five-question quiz based on the lesson.",
        backstory=(
            "You create fair practice questions. Make exactly five numbered questions: "
            "three knowledge questions and two application questions. Do not reveal answers."
        ),
        llm=llm, verbose=True, allow_delegation=False, max_iter=3,
    )
    evaluator = Agent(
        role="Answer Evaluator",
        goal="Evaluate the student's answers and provide useful feedback.",
        backstory=(
            "You are fair, precise, and encouraging. Evaluate each answer against the "
            "lesson, give a score out of five, explain mistakes, and identify the weakest "
            "concept. Include a clearly formatted line: Score: X/5."
        ),
        llm=llm, verbose=True, allow_delegation=False, max_iter=3,
    )
    return coordinator, explainer, quiz_master, evaluator

coordinator, explainer, quiz_master, evaluator = build_agents()
print("All four agents created.")


## 6. Coordinator → Explainer → Quiz Master
This is a sequential Crew. Task context passes the Coordinator's plan to the Explainer and the plan + lesson to the Quiz Master. Uses native async `akickoff` for Colab.

In [ ]:
async def run_learning_phase(student_name, level, topic, previous_context=""):
    coordinator, explainer, quiz_master, _ = build_agents()

    plan_task = Task(
        description=(
            "Student: {student_name}\nLevel: {level}\nTopic: {topic}\n"
            "Previous learning context: {previous_context}\n"
            "Create a concise plan with 3-4 learning objectives, prerequisites, "
            "and recommended teaching order. This plan is handed to the Explainer."
        ),
        expected_output="A numbered learning plan with 3-4 objectives and teaching order.",
        agent=coordinator,
    )
    teach_task = Task(
        description=(
            "Teach {topic} to {student_name}, whose level is {level}. Use the Coordinator's "
            "plan from task context. Explain key concepts simply, include one worked example, "
            "and finish with a short recap. Your lesson is handed to Quiz Master."
        ),
        expected_output="A clear lesson with definitions, example, and recap.",
        agent=explainer,
        context=[plan_task],
    )
    quiz_task = Task(
        description=(
            "Create exactly five practice questions for {student_name} on {topic}. "
            "Use the Coordinator's plan and Explainer's lesson from task context. Include "
            "three knowledge questions and two application questions. Do not show answers."
        ),
        expected_output="Five numbered questions without an answer key.",
        agent=quiz_master,
        context=[plan_task, teach_task],
    )

    crew = Crew(
        agents=[coordinator, explainer, quiz_master],
        tasks=[plan_task, teach_task, quiz_task],
        process=Process.sequential,
        memory=False,
        verbose=True,
    )
    result = await crew.akickoff(inputs={
        "student_name": student_name,
        "level": level,
        "topic": topic,
        "previous_context": previous_context or "No previous session context.",
    })
    outputs = result.tasks_output
    if len(outputs) < 3:
        raise RuntimeError("CrewAI did not return all three learning task outputs.")
    return outputs[0].raw, outputs[1].raw, outputs[2].raw


## 7. Start the learning session

In [ ]:
saved = load_memory()
student_name = input(f"Student name [{saved.get('student_name', 'Student')}]: ").strip()
student_name = student_name or saved.get("student_name", "Student")
level = input("Level (beginner/intermediate/advanced) [beginner]: ").strip() or "beginner"
topic = input(f"Topic [{saved.get('last_topic', 'Python Basics')}]: ").strip()
topic = topic or saved.get("last_topic", "Python Basics")
previous_context = f"Previously studied: {saved.get('last_topic')}" if saved.get("last_topic") else ""

print("\\nStarting Coordinator -> Explainer -> Quiz Master...")
plan, lesson, quiz = await run_learning_phase(student_name, level, topic, previous_context)

print("\\n" + "="*68 + "\\nCOORDINATOR: LEARNING PLAN\\n" + "="*68)
print(plan)
print("\\n" + "="*68 + "\\nEXPLAINER: LESSON\\n" + "="*68)
print(lesson)
print("\\n" + "="*68 + "\\nQUIZ MASTER: QUIZ\\n" + "="*68)
print(quiz)


## 8. Quiz Master → Evaluator handoff

In [ ]:
async def evaluate_answers(student_name, topic, lesson, quiz, answers):
    _, _, _, evaluator = build_agents()
    evaluation_task = Task(
        description=(
            "Evaluate the student's quiz answers. Student: {student_name}; Topic: {topic}\n"
            "LESSON:\n{lesson}\nQUIZ:\n{quiz}\nSTUDENT ANSWERS:\n{answers}\n\n"
            "For each of five questions, mark correct, partly correct, or incorrect and "
            "explain briefly. Give a total score out of five. Include a line exactly in "
            "this format: Score: X/5. Then list strengths, weakest concept, and review advice."
        ),
        expected_output="Question-by-question assessment, Score: X/5, strengths, and review advice.",
        agent=evaluator,
    )
    crew = Crew(
        agents=[evaluator], tasks=[evaluation_task],
        process=Process.sequential, memory=False, verbose=True,
    )
    result = await crew.akickoff(inputs={
        "student_name": student_name,
        "topic": topic,
        "lesson": lesson,
        "quiz": quiz,
        "answers": answers,
    })
    return result.tasks_output[0].raw

answers = input("Write your answers (number them 1-5): ").strip()
if not answers:
    answers = "The student did not submit answers."

print("\\nQuiz Master -> Evaluator: passing lesson, quiz, and student answers...")
feedback = await evaluate_answers(student_name, topic, lesson, quiz, answers)
print("\\n" + "="*68 + "\\nEVALUATOR: FEEDBACK\\n" + "="*68)
print(feedback)


## 9. Bonus +10: Evaluator feedback → Explainer re-teaching

In [ ]:
def score_from_feedback(feedback_text):
    match = re.search(
        r"Score\\s*:\\s*(\\d+(?:\\.\\d+)?)\\s*/\\s*5",
        feedback_text,
        flags=re.IGNORECASE
    )
    return float(match.group(1)) if match else None

async def re_teach_weak_area(student_name, level, topic, feedback):
    _, explainer, _, _ = build_agents()
    reteach_task = Task(
        description=(
            "Re-teach the weakest concept identified in the evaluator feedback.\n"
            "Student: {student_name}\nLevel: {level}\nTopic: {topic}\n"
            "Evaluator feedback: {feedback}\n"
            "Explain the weak concept in a simpler, different way. Include a fresh analogy "
            "or example and finish with two short check questions."
        ),
        expected_output="A focused remedial mini-lesson and two check questions.",
        agent=explainer,
    )
    crew = Crew(
        agents=[explainer], tasks=[reteach_task],
        process=Process.sequential, memory=False, verbose=True,
    )
    result = await crew.akickoff(inputs={
        "student_name": student_name,
        "level": level,
        "topic": topic,
        "feedback": feedback,
    })
    return result.tasks_output[0].raw

score = score_from_feedback(feedback)
if score is not None and score < 4:
    print(f"Score {score}/5: starting Evaluator -> Explainer re-teaching loop.")
    remedial = await re_teach_weak_area(student_name, level, topic, feedback)
    print("\\n" + "="*68 + "\\nEXPLAINER: REMEDIAL LESSON\\n" + "="*68)
    print(remedial)
elif score is None:
    print("Score format not detected. Review evaluator feedback manually; loop not auto-triggered.")
else:
    print(f"Score {score}/5: re-teaching loop not needed.")


## 10. Save and verify student memory

In [ ]:
save_memory(student_name, topic, level, score, feedback)
print("Memory saved to:", MEMORY_FILE)
print(json.dumps(load_memory(), ensure_ascii=False, indent=2))


## 11. Assignment checklist

- [x] Four distinct agents, each with its own role, goal, and backstory prompt
- [x] CrewAI framework (allowed by the assignment)
- [x] Sequential orchestration
- [x] Coordinator plan handed to Explainer
- [x] Plan and lesson handed to Quiz Master
- [x] Quiz, lesson, and student answers handed to Evaluator
- [x] Student name and topic remembered in `leo_memory.json`
- [x] Bonus feedback loop: Evaluator feedback is handed to Explainer for re-teaching
- [x] Secure API-key input and preflight API test

**Demo video:** Record a 3–5 minute screen video showing one complete session, each agent's turn, the handoffs, quiz, evaluation, and (if triggered) re-teaching. The video must be recorded by the student.
